# 01 · Baselines: Poisson GLM and GBM
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/actuarialsociety/deep-actuarial-science/blob/main/notebooks/01_glm_gbm_baselines.ipynb)

Claim frequency $N_i \sim \mathrm{Poi}(v_i\lambda(x_i))$ on freMTPL2. Every deep model later in the course has to beat these two.

_Runs on a Colab **GPU runtime** (T4), selected automatically when you open it; if Colab gives you a CPU runtime, the first cell stops and says how to switch._

In [ ]:
# Setup: works in Colab (GPU runtime), the dev container and a local clone.
import os, sys
# Colab must run on a GPU runtime: stop here otherwise (set DAS_ALLOW_CPU=1 to override).
if "google.colab" in sys.modules and os.environ.get("DAS_ALLOW_CPU") != "1":
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError(
            "This notebook needs a GPU runtime. In Colab: Runtime > Change runtime type > "
            "T4 GPU > Save, then Runtime > Run all. (To run on CPU anyway: "
            "import os; os.environ['DAS_ALLOW_CPU'] = '1')"
        )
    print("GPU:", torch.cuda.get_device_name(0))

if "google.colab" in sys.modules and not os.path.exists("/content/deep-actuarial-science"):
    !git clone -q https://github.com/actuarialsociety/deep-actuarial-science /content/deep-actuarial-science
    !pip install -q -e /content/deep-actuarial-science
if "google.colab" in sys.modules:
    os.chdir("/content/deep-actuarial-science")
    sys.path.insert(0, "src")
os.environ.setdefault("DO_NOT_TRACK", "1")
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")

from deep_actuarial_science.notebook import run, load, show, leaderboard, parameters, mlflow_ui
FAST = os.environ.get("DAS_ENV") == "smoke"   # CI runs every notebook in smoke mode


In [ ]:
models = ["homogeneous", "glm", "gbm"]
run("frequency", models=models)
leaderboard("frequency", models)

## Open the box: GLM coefficients
The same `GLM` class the pipeline uses:

In [ ]:
glm = load("glm.freq_model")
glm.result.summary().tables[1]

## Diagnostics (test set)
Calibration by decile, marginal fits and the GBM's early-stopping curve.

In [ ]:
show("frequency", "glm", ["calibration", "marginal_drivage"])
show("frequency", "gbm", ["marginal_drivage", "training"])

## Exercise
Add `BonusMalusGLM + VehAgeGLM` to the GLM formula (see Quickstart §3). How close does the GLM get to the GBM? Which feature engineering closes the gap?

## Explore the runs in MLflow
Parameters, metrics, figures and the benchmark table of every run above. In Colab the UI opens below and in a new tab through Colab's proxy (MLflow runs on the Colab machine, so a plain localhost link would point at your own computer).

In [ ]:
mlflow_ui()